In [ ]:
import pandas as pd
from scipy.stats import shapiro, ttest_ind, chi2_contingency, fisher_exact, mannwhitneyu
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
from statsmodels.stats.multitest import multipletests
import statsmodels.api as sm
from IPython.display import display


df = pd.read_excel("synthetic_data.xlsx", sheet_name = "PP_Data")

In [ ]:
# Test normality and compare continuous variables between groups

variables = ["Age", "Height", "Weight", "BMI"]

for var in variables:
    control = df[df["Group"] == 0][var]
    probiotic = df[df["Group"] == 1][var]

    pcontrol = shapiro(control).pvalue
    pprobiotic = shapiro(probiotic).pvalue

    print(f"\n{var}")
    print(f"Control Shapiro p = {pcontrol:.4f}")
    print(f"Probiotic Shapiro p = {pprobiotic:.4f}")


    if pcontrol > 0.05 and pprobiotic > 0.05:
        stat, p = ttest_ind(control, probiotic)
        test_name = "t-test"
    else:
        stat, p = mannwhitneyu(control, probiotic, alternative="two-sided")
        test_name = "Mann-Whitney U"

    print(f"{test_name}: p = {p:.4f}")

In [ ]:
# Calculating the median, quartiles, P-value, and effect size

results = []

for var in variables:
    control = df[df["Group"] == 0][var]
    probiotic = df[df["Group"] == 1][var]

    # Mann-Whitney U test
    U, p = mannwhitneyu(
        control,
        probiotic,
        alternative="two-sided",
        method="asymptotic")

    # Sample sizes
    n1 = len(control)
    n2 = len(probiotic)
    N = n1 + n2

    # Mean of U under the null hypothesis
    mean_U = n1 * n2 / 2

    # Tie correction
    combined = np.concatenate([control, probiotic])
    _, tie_counts = np.unique(combined, return_counts=True)

    tie_term = np.sum(tie_counts**3 - tie_counts)

    # Variance of U with tie correction
    variance_U = (
        n1 * n2 / 12
        * (
            N + 1
            - tie_term / (N * (N - 1))
        ))

    sd_U = np.sqrt(variance_U)

    # Continuity-corrected Z
    if U > mean_U:
        z = (U - mean_U - 0.5) / sd_U
    elif U < mean_U:
        z = (U - mean_U + 0.5) / sd_U
    else:
        z = 0


    # Effect size (r)
    r = abs(z) / np.sqrt(N)

    # Median and IQR
    c_med = control.median()
    p_med = probiotic.median()

    c_q1 = control.quantile(0.25)
    c_q3 = control.quantile(0.75)

    p_q1 = probiotic.quantile(0.25)
    p_q3 = probiotic.quantile(0.75)

    results.append([
        var,
        f"{c_med:.2f} ({c_q1:.2f}–{c_q3:.2f})",
        f"{p_med:.2f} ({p_q1:.2f}–{p_q3:.2f})",
        round(p, 3),
        round(r, 3)
    ])

demographic_table = pd.DataFrame(
    results,
    columns=[
        "Variable",
        "Control Median (Q1–Q3)",
        "Probiotic Median (Q1–Q3)",
        "P-value",
        "Effect size (r)"
    ])

demographic_table

In [ ]:
# Gender distribution among two groups

table = pd.crosstab(df["Gender"], df["Group"])

chi2, p, dof, expected = chi2_contingency(table, correction=False)

results =[]

for sexcode, sexname in [(0,"Male"),(1,"Female")]:
    controlcount = table.loc[sexcode,0]
    probioticcount = table.loc[sexcode,1]

    controlpercent = 100*controlcount/table[0].sum()
    probioticpercent = 100*probioticcount/table[1].sum()

    results.append([sexname,f"{controlcount}({controlpercent:.1f}%)",f"{probioticcount}({probioticpercent:.1f}%)",round(p,3)])

gender_table = pd.DataFrame(results,columns=["sex","Control n (%)","Probiotic n (%)","P_value"])
gender_table

In [ ]:
#Chief complaints table

df['CC'] = df['CC'].fillna('').astype(str)

symptom_codes = {
    0: 'Other',
    1: 'Heartburn',
    2: 'GIB',
    3: 'Dyspepsia',
    4: 'Diarrhea'
}

results = []

for code, name in symptom_codes.items():

    present = (',' + df['CC'] + ',').str.contains(f',{code},')

    table = pd.crosstab(df['Group'], present)

    if table.shape != (2, 2):
        continue

    chi2, chi_p, _, expected = chi2_contingency(table,correction=False)

    if (expected < 5).any():
        _, p = fisher_exact(table)
        test_name = "Fisher's exact"
    else:
        p = chi_p
        test_name = "Chi-square"

    control_n = (df['Group'] == 0).sum()
    probiotic_n = (df['Group'] == 1).sum()

    control_count = ((df['Group'] == 0) & present).sum()
    probiotic_count = ((df['Group'] == 1) & present).sum()

    total_count = control_count + probiotic_count

    results.append([
        name,
        f"{control_count} ({100 * control_count / control_n:.1f}%)",
        f"{probiotic_count} ({100 * probiotic_count / probiotic_n:.1f}%)",
        p,
        test_name,
        total_count
    ])


#Final table

final_results = []

for i in range(len(results)):
    final_results.append([
        results[i][0],
        results[i][1],
        results[i][2],
        round(results[i][3], 4),
        results[i][4],
        results[i][5]
    ])

CC_table = pd.DataFrame(
    final_results,
    columns=[
        'Chief Complaint',
        'Control',
        'Probiotic',
        'P-value',
        'Test',
        'Total'
    ]
)


# Sort by total frequency: highest to lowest

CC_table = CC_table.sort_values(
    'Total',
    ascending=False
).reset_index(drop=True)


# Remove Total from the displayed table

CC_table = CC_table.drop(columns='Total')

CC_table

In [ ]:
# Past medical history table

df['PMH'] = df['PMH'].fillna('').astype(str)

diseases = {
    1: "DM",
    2: "GI disease",
    3: "CVD",
    4: "HTN",
    5: "Thyroid disease"
}

results = []

for code, name in diseases.items():

    present = (',' + df['PMH'] + ',').str.contains(f',{code},')

    table = pd.crosstab(df['Group'], present)

    if table.shape != (2, 2):
        continue

    chi2, chi_p, _, expected = chi2_contingency(table,correction=False)

    if (expected < 5).any():
        _, p = fisher_exact(table)
        test_name = "Fisher's exact"
    else:
        p = chi_p
        test_name = "Chi-square"

    control_n = (df['Group'] == 0).sum()
    probiotic_n = (df['Group'] == 1).sum()

    control_count = ((df['Group'] == 0) & present).sum()
    probiotic_count = ((df['Group'] == 1) & present).sum()

    total_count = control_count + probiotic_count

    results.append([
        name,
        f"{control_count} ({100 * control_count / control_n:.1f}%)",
        f"{probiotic_count} ({100 * probiotic_count / probiotic_n:.1f}%)",
        p,
        test_name,
        total_count
    ])


# Create final table

final_results = []

for i in range(len(results)):
    final_results.append([
        results[i][0],
        results[i][1],
        results[i][2],
        round(results[i][3], 4),
        results[i][4],
        results[i][5]
    ])

PMH_table = pd.DataFrame(
    final_results,
    columns=[
        "Disease",
        "Control",
        "Probiotic",
        "P-value",
        "Test",
        "Total"
    ]
)


# Sort by total frequency: highest to lowest

PMH_table = PMH_table.sort_values(
    "Total",
    ascending=False
).reset_index(drop=True)


# Remove Total from the displayed table

PMH_table = PMH_table.drop(columns="Total")

PMH_table

In [ ]:
#Lifestyle factors table

variables = ["Alcohol", "Coffee", "Smoking"]

results = []

for v in variables:

    table = pd.crosstab(df[v], df["Group"])

    if table.shape != (2, 2):
        continue

    chi2, chi_p, _, expected = chi2_contingency(
        table,
        correction=False
    )

    if (expected < 5).any():
        _, p = fisher_exact(table)
        test_name = "Fisher's exact"
    else:
        p = chi_p
        test_name = "Chi-square"

    control_n = (df["Group"] == 0).sum()
    probiotic_n = (df["Group"] == 1).sum()

    control_count = ((df["Group"] == 0) & (df[v] == 1)).sum()
    probiotic_count = ((df["Group"] == 1) & (df[v] == 1)).sum()

    total_count = control_count + probiotic_count

    results.append([
        v,
        f"{control_count} ({100 * control_count / control_n:.1f}%)",
        f"{probiotic_count} ({100 * probiotic_count / probiotic_n:.1f}%)",
        p,
        test_name,
        total_count
    ])


# Create final table

final_results = []

for i in range(len(results)):
    final_results.append([
        results[i][0],
        results[i][1],
        results[i][2],
        round(results[i][3], 4),
        results[i][4],
        results[i][5]
    ])

lifestyle_table = pd.DataFrame(
    final_results,
    columns=[
        "Variable",
        "Control n (%)",
        "Probiotic n (%)",
        "P-value",
        "Test",
        "Total"
    ]
)


# Sort by total frequency: highest to lowest

lifestyle_table = lifestyle_table.sort_values(
    "Total",
    ascending=False
).reset_index(drop=True)


# Remove Total from displayed table

lifestyle_table = lifestyle_table.drop(columns="Total")

lifestyle_table

In [ ]:
#Primary and secondary outcomes

results = []

# 1. Treatment Success (PP)

a = ((df["Group"] == 1) & (df["Stool_Ag_Test"] == 0)).sum()
b = ((df["Group"] == 1) & (df["Stool_Ag_Test"] != 0)).sum()

c = ((df["Group"] == 0) & (df["Stool_Ag_Test"] == 0)).sum()
d = ((df["Group"] == 0) & (df["Stool_Ag_Test"] != 0)).sum()

table = [[a, b], [c, d]]

chi2, chi_p, _, expected = chi2_contingency(table, correction = False)

if (expected < 5).any():
    _, p = fisher_exact(table)
    test_name = "Fisher's exact"
else:
    p = chi_p
    test_name = "Chi-square"

rr = (a / (a + b)) / (c / (c + d))

se = np.sqrt(
    (1 / a) - (1 / (a + b)) +
    (1 / c) - (1 / (c + d)))

lower = np.exp(np.log(rr) - 1.96 * se)
upper = np.exp(np.log(rr) + 1.96 * se)

results.append([
    "Treatment success",
    "Per-protocol",
    f"{a}/{a+b} ({100*a/(a+b):.1f}%)",
    f"{c}/{c+d} ({100*c/(c+d):.1f}%)",
    round(rr, 3),
    f"{lower:.3f} - {upper:.3f}",
    round(p, 4),
    test_name])


# 2. Treatment Success (ITT)
# Lost to follow-up considered treatment failures (considering lost to follow-up is 7 and 2 in probiotic and placebo)
b_itt = b + 7
d_itt = d + 2

table = [[a, b_itt], [c, d_itt]]

chi2, chi_p, _, expected = chi2_contingency(table, correction = False)

if (expected < 5).any():
    _, p = fisher_exact(table)
    test_name = "Fisher's exact"
else:
    p = chi_p
    test_name = "Chi-square"

rr = (a / (a + b_itt)) / (c / (c + d_itt))

se = np.sqrt(
    (1 / a) - (1 / (a + b_itt)) +
    (1 / c) - (1 / (c + d_itt))
)

lower = np.exp(np.log(rr) - 1.96 * se)
upper = np.exp(np.log(rr) + 1.96 * se)

results.append([
    "Treatment success",
    "ITT",
    f"{a}/{a+b_itt} ({100*a/(a+b_itt):.1f}%)",
    f"{c}/{c+d_itt} ({100*c/(c+d_itt):.1f}%)",
    round(rr, 3),
    f"{lower:.3f} - {upper:.3f}",
    round(p, 4),
    test_name])


#3. Complications (PP)

a_com = ((df["Group"] == 1) & (df["Complication_Index"] == 1)).sum()
b_com = ((df["Group"] == 1) & (df["Complication_Index"] != 1)).sum()

c_com = ((df["Group"] == 0) & (df["Complication_Index"] == 1)).sum()
d_com = ((df["Group"] == 0) & (df["Complication_Index"] != 1)).sum()

table = [[a_com, b_com], [c_com, d_com]]

chi2, chi_p, _, expected = chi2_contingency(table, correction = False)

if (expected < 5).any():
    _, p = fisher_exact(table)
    test_name = "Fisher's exact"
else:
    p = chi_p
    test_name = "Chi-square"

rr = (a_com / (a_com + b_com)) / (c_com / (c_com + d_com))

se = np.sqrt(
    (1 / a_com) - (1 / (a_com + b_com)) +
    (1 / c_com) - (1 / (c_com + d_com))
)

lower = np.exp(np.log(rr) - 1.96 * se)
upper = np.exp(np.log(rr) + 1.96 * se)

results.append([
    "Complications",
    "Per-protocol",
    f"{a_com}/{a_com+b_com} ({100*a_com/(a_com+b_com):.1f}%)",
    f"{c_com}/{c_com+d_com} ({100*c_com/(c_com+d_com):.1f}%)",
    round(rr, 3),
    f"{lower:.3f} - {upper:.3f}",
    round(p, 4),
    test_name])


#Final Table

rr_table = pd.DataFrame(
    results,
    columns=[
        "Outcome",
        "Analysis",
        "Probiotic n (%)",
        "Control n (%)",
        "RR",
        "95% CI",
        "P-value",
        "Test"])

rr_table

In [ ]:
# Bar chart for primary and secondary outcomes
# Get values from rr_table

# Treatment Success - ITT
success_itt = rr_table.loc[
    (rr_table["Outcome"] == "Treatment success") &
    (rr_table["Analysis"] == "ITT"),
    "P-value"
].iloc[0]

# Treatment Success - PP
success_pp = rr_table.loc[
    (rr_table["Outcome"] == "Treatment success") &
    (rr_table["Analysis"] == "Per-protocol"),
    "P-value"
].iloc[0]

# Complications - PP
complication_p = rr_table.loc[
    (rr_table["Outcome"] == "Complications") &
    (rr_table["Analysis"] == "Per-protocol"),
    "P-value"
].iloc[0]


# Percentages
success_values = [
    100 * c / (c + d_itt),      # Control ITT
    100 * a / (a + b_itt),      # S. boulardii ITT
    100 * c / (c + d),          # Control PP
    100 * a / (a + b)           # S. boulardii PP
]

complication_values = [
    100 * c_com / (c_com + d_com),   # Control PP
    100 * a_com / (a_com + b_com)    # S. boulardii PP
]

# Colors
control_color = "lightgray"
probiotic_color = "dimgray"

# Figure
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
bar_width = 0.55

# Panel 1: H. pylori Eradication
x1 = np.array([0, 0.7, 2, 2.7])

colors_success = [
    control_color,
    probiotic_color,
    control_color,
    probiotic_color
]

axes[0].bar(
    x1,
    success_values,
    width=bar_width,
    color=colors_success
)

axes[0].set_title("A")
axes[0].set_ylabel("Participants (%)")
axes[0].set_ylim(0, 110)
axes[0].set_xlim(-0.5, 3.2)

axes[0].set_xticks(x1)
axes[0].set_xticklabels([
    "Placebo",
    "S. boulardii",
    "Placebo",
    "S. boulardii"
])


# Analysis labels
axes[0].text(
    0.35,
    106,
    "ITT",
    ha="center",
    fontsize=11,
    fontweight="bold"
)

axes[0].text(
    2.35,
    106,
    "PP",
    ha="center",
    fontsize=11,
    fontweight="bold"
)


# P-values
axes[0].text(
    0.35,
    99,
    f"P = {success_itt:.4f}",
    ha="center",
    fontsize=10
)

axes[0].text(
    2.35,
    99,
    f"P = {success_pp:.4f}",
    ha="center",
    fontsize=10
)


# Percentage labels
for i, value in enumerate(success_values):
    axes[0].text(
        x1[i],
        value + 2,
        f"{value:.1f}%",
        ha="center"
    )

# Panel 2: Treatment-related Adverse Events
x2 = np.array([1, 1.7])

colors_complication = [
    control_color,
    probiotic_color
]

axes[1].bar(
    x2,
    complication_values,
    width=bar_width,
    color=colors_complication
)

axes[1].set_title("B")
axes[1].set_ylabel("Participants (%)")
axes[1].set_ylim(0, 110)
axes[1].set_xlim(-0.5, 3.2)

axes[1].set_xticks(x2)
axes[1].set_xticklabels([
    "Placebo",
    "S. boulardii"
])


# P-value
if complication_p < 0.001:
    complication_p_text = "P < 0.001"
else:
    complication_p_text = f"P = {complication_p:.4f}"

axes[1].text(
    1.35,
    99,
    complication_p_text,
    ha="center",
    fontsize=10
)


# Percentage labels
for i, value in enumerate(complication_values):
    axes[1].text(
        x2[i],
        value + 2,
        f"{value:.1f}%",
        ha="center"
    )

# legend
legend_elements = [Patch(
        facecolor=control_color,
        label="Placebo"
    ),
    Patch(
        facecolor=probiotic_color,
        label="S. boulardii"
    )
]

fig.legend(
    handles=legend_elements,
    loc="upper center",
    ncol=2,
    bbox_to_anchor=(0.5, 1.02),
    frameon=False
)


# Final layout
plt.tight_layout()
plt.savefig("figure1.pdf", bbox_inches="tight")
plt.savefig("figure1.tif", dpi=600, bbox_inches="tight")
plt.show()

In [ ]:
# Treatment-related adverse events

results = []
p_values = []

side_effects = {
    'تلخی دهان': 'Bitter taste',
    'مدفوع سیاه': 'Black stool',
    'تهوع': 'Nausea',
    'درد معده': 'Epigastric pain',
    'اسهال': 'Diarrhea',
    'نفخ شکم': 'Bloating',
    'یبوست': 'Constipation'
}

control_n = (df["Group"] == 0).sum()
probiotic_n = (df["Group"] == 1).sum()


for persian, english in side_effects.items():

    present = df["Complication"].astype(str).str.contains(
        persian,
        na=False
    )

    # Counts
    a = ((df["Group"] == 1) & present).sum()
    b = ((df["Group"] == 1) & (~present)).sum()
    c = ((df["Group"] == 0) & present).sum()
    d = ((df["Group"] == 0) & (~present)).sum()

    table = [[a, b], [c, d]]

    # Chi-square or Fisher's exact test
    chi2, chi_p, _, expected = chi2_contingency(
        table,
        correction=False
    )

    if (expected < 5).any():
        _, p = fisher_exact(table)
        test_used = "Fisher's exact"
    else:
        p = chi_p
        test_used = "Chi-square"

    # Percentages
    probiotic_pct = 100 * a / probiotic_n
    control_pct = 100 * c / control_n

    # RR and 95% CI
    # Apply a 0.5 continuity correction only if an event count is zero.
    a2, b2, c2, d2 = a, b, c, d

    if a2 == 0 or c2 == 0:
        a2 += 0.5
        b2 += 0.5
        c2 += 0.5
        d2 += 0.5

    rr = (a2 / (a2 + b2)) / (c2 / (c2 + d2))

    se = np.sqrt(
        (1 / a2) - (1 / (a2 + b2)) +
        (1 / c2) - (1 / (c2 + d2))
    )

    lower = np.exp(np.log(rr) - 1.96 * se)
    upper = np.exp(np.log(rr) + 1.96 * se)

    total_count = a + c

    results.append({
        "Side Effect": english,
        "Probiotic n (%)": f"{a} ({probiotic_pct:.1f}%)",
        "Control n (%)": f"{c} ({control_pct:.1f}%)",
        "Test": test_used,
        "RR": rr,
        "95% CI": f"{lower:.3f} - {upper:.3f}",
        "Unadjusted P-value": p,
        "Total": total_count
    })

    p_values.append(p)

# FDR adjustment
p_values_arr = np.array(p_values)

valid_idx = ~np.isnan(p_values_arr)

reject, pvals_fdr, _, _ = multipletests(
    p_values_arr[valid_idx],
    alpha=0.05,
    method="fdr_bh"
)

fdr_idx = 0

for i in range(len(results)):

    raw_p = results[i]["Unadjusted P-value"]

    if not np.isnan(raw_p):
        results[i]["FDR-adjusted P-value"] = round(
            pvals_fdr[fdr_idx],
            4
        )
        fdr_idx += 1
    else:
        results[i]["FDR-adjusted P-value"] = "-"

# Final Table
complication_table = pd.DataFrame(results)

# Sort by total frequency: highest to lowest
complication_table = complication_table.sort_values(
    "Total",
    ascending=False
).reset_index(drop=True)

# Formatting
complication_table["RR"] = complication_table["RR"].round(3)
complication_table["Unadjusted P-value"] = (
    complication_table["Unadjusted P-value"].round(4)
)

# Remove sorting column from displayed table
complication_table = complication_table.drop(columns="Total")

complication_table

In [ ]:
# Forest plot for treatment-related adverse events
# Prepare data from complication_table

forest_data = complication_table.copy()

forest_data[["CI_lower", "CI_upper"]] = (
    forest_data["95% CI"]
    .str.split(" - ", expand=True)
    .astype(float)
)

forest_data["RR"] = forest_data["RR"].astype(float)


# Figure layout

fig = plt.figure(figsize=(12.5, 6.5))

gs = fig.add_gridspec(
    1,
    3,
    width_ratios=[1.8, 2.8, 1.5],
    wspace=0.02
)

ax_table = fig.add_subplot(gs[0, 0])
ax_plot = fig.add_subplot(gs[0, 1])
ax_values = fig.add_subplot(gs[0, 2])


# Y positions

n = len(forest_data)

y = np.arange(n)

for ax in [ax_table, ax_plot, ax_values]:
    ax.set_ylim(-1, n)

ax_table.invert_yaxis()
ax_plot.invert_yaxis()
ax_values.invert_yaxis()


# LEFT TABLE
ax_table.axis("off")

# Column positions
x_event = 0.00
x_probiotic = 0.55
x_placebo = 0.92


# Headers
ax_table.text(
    x_event,
    -0.55,
    "Adverse event",
    fontsize=10,
    fontweight="bold",
    va="center"
)

ax_table.text(
    x_probiotic,
    -0.55,
    "S. boulardii",
    fontsize=9,
    fontweight="bold",
    ha="center",
    va="center"
)

ax_table.text(
    x_placebo,
    -0.55,
    "Placebo",
    fontsize=9,
    fontweight="bold",
    ha="center",
    va="center"
)


# Rows
for i, row in forest_data.iterrows():

    ax_table.text(
        x_event,
        i,
        row["Side Effect"],
        fontsize=9.5,
        va="center"
    )

    ax_table.text(
        x_probiotic,
        i,
        row["Probiotic n (%)"],
        fontsize=8.5,
        ha="center",
        va="center"
    )

    ax_table.text(
        x_placebo,
        i,
        row["Control n (%)"],
        fontsize=8.5,
        ha="center",
        va="center"
    )

# CENTER FOREST PLOT
for i, row in forest_data.iterrows():

    ax_plot.plot(
        [row["CI_lower"], row["CI_upper"]],
        [i, i],
        color="black",
        linewidth=1.5,
        solid_capstyle="round",
        zorder=2
    )

    ax_plot.scatter(
        row["RR"],
        i,
        marker="s",
        s=55,
        color="black",
        zorder=3
    )


# Reference line
ax_plot.axvline(
    1,
    color="gray",
    linestyle="--",
    linewidth=1
)

# Logarithmic X-axis
ax_plot.set_xscale("log")

ax_plot.set_xlim(0.005, 100)

ax_plot.set_xticks([
    0.01,
    0.1,
    1,
    10,
    100
])

ax_plot.set_xticklabels([
    "0.01",
    "0.1",
    "1",
    "10",
    "100"
])

ax_plot.set_yticks(y)
ax_plot.set_yticklabels([])

ax_plot.set_xlabel(
    "Risk Ratio",
    fontsize=10
)

# RIGHT: RR AND 95% CI
ax_values.axis("off")

ax_values.text(
    0.00,
    -0.55,
    "RR (95% CI)",
    fontsize=10,
    fontweight="bold",
    va="center"
)

for i, row in forest_data.iterrows():

    ax_values.text(
        0.00,
        i,
        f'{row["RR"]:.3f} ({row["CI_lower"]:.3f}–{row["CI_upper"]:.3f})',
        fontsize=8.5,
        va="center"
    )

# Grid
ax_plot.grid(
    axis="x",
    linestyle=":",
    linewidth=0.7,
    alpha=0.5
)

ax_plot.grid(
    axis="y",
    visible=False
)

# Clean appearance
ax_plot.spines["top"].set_visible(False)
ax_plot.spines["right"].set_visible(False)
ax_plot.spines["left"].set_visible(False)

# Final layout
plt.subplots_adjust(
    left=0.05,
    right=0.98,
    top=0.87,
    bottom=0.15
)
plt.savefig("figure2.pdf", bbox_inches="tight")
plt.savefig("figure2.tif", dpi=600, bbox_inches="tight")
plt.show()

In [ ]:
#Logistic regression (reduced version)
def logistic_table(outcome):

    # Primary adjusted model
    predictors = [
        "Group",
        "Age",
        "Gender",
        "BMI"
    ]

    X = df[predictors].copy()

    X = sm.add_constant(X)

    # Outcome definition
    if outcome == "Stool_Ag_Test":
        # 0 = negative stool antigen = treatment success
        y = (df[outcome] == 0).astype(int)

    elif outcome == "Complication_Index":
        y = df[outcome].astype(int)

    else:
        raise ValueError("Unknown outcome variable.")

    # Logistic regression
    model = sm.Logit(y, X).fit(disp=False)

    beta = model.params
    se = model.bse
    p = model.pvalues

    OR = np.exp(beta)
    lower = np.exp(beta - 1.96 * se)
    upper = np.exp(beta + 1.96 * se)

    table = pd.DataFrame({
        "Variable": OR.index,
        "Adjusted OR": OR.values,
        "95% CI": [
            f"{l:.3f} - {u:.3f}"
            for l, u in zip(lower, upper)
        ],
        "P-value": p.values
    })

    table = table.round({
        "Adjusted OR": 3,
        "P-value": 4
    })

    table = table[table["Variable"] != "const"].reset_index(drop=True)

    return table


success_log = logistic_table("Stool_Ag_Test")
complication_log = logistic_table("Complication_Index")


print("Treatment Success")
display(success_log)

print("\nComplications")
display(complication_log)

In [ ]:
# Logistic regression (extended)
df["PMH"] = df["PMH"].astype(str)

df["DM"] = ("," + df["PMH"] + ",").str.contains(",1,").astype(int)
df["GI_disease"] = ("," + df["PMH"] + ",").str.contains(",2,").astype(int)
df["HTN"] = ("," + df["PMH"] + ",").str.contains(",4,").astype(int)
df["Thyroid"] = ("," + df["PMH"] + ",").str.contains(",5,").astype(int)


def logistic_table(outcome):

    predictors = ["Group",
        "Age",
        "BMI",
        "Gender",
        "Smoking",
        "Coffee",
        "DM",
        "GI_disease",
        "HTN",
        "Thyroid"]

    X = df[predictors].copy()
    X = sm.add_constant(X)

   
    if outcome == "Stool_Ag_Test":
        # 0 = negative stool antigen = treatment success
        # Recode so that success = 1
        y = (df[outcome] == 0).astype(int)

    elif outcome == "Complication_Index":
        y = df[outcome].astype(int)

    else:
        raise ValueError("Unknown outcome variable.")


    model = sm.Logit(y, X).fit(disp=False)

    beta = model.params
    se = model.bse
    p = model.pvalues

    OR = np.exp(beta)
    lower = np.exp(beta - 1.96 * se)
    upper = np.exp(beta + 1.96 * se)

    table = pd.DataFrame({"Variable": OR.index,
        "Adjusted OR": OR.values,
        "95% CI": [
            f"{l:.3f} - {u:.3f}"
            for l, u in zip(lower, upper)],
        "P-value": p.values})

    table = table.round({"Adjusted OR": 3,
        "P-value": 4})

   
    table = table[table["Variable"] != "const"].reset_index(drop=True)

    return table



success_logistic = logistic_table("Stool_Ag_Test")
complication_logistic = logistic_table("Complication_Index")

print("Treatment Success")
display(success_logistic)

print("\nComplications")
display(complication_logistic)

In [ ]:
with pd.ExcelWriter("Results.xlsx", engine="openpyxl") as writer:

    demographic_table.to_excel(writer, sheet_name="Baseline Characteristics", index=False)

    gender_table.to_excel(writer, sheet_name="Gender Distribution", index=False)

    CC_table.to_excel(writer, sheet_name="Chief Complaint", index=False)

    PMH_table.to_excel(writer, sheet_name="Comorbidities", index=False)

    lifestyle_table.to_excel(writer, sheet_name="Life Style", index=False)

    rr_table.to_excel(writer, sheet_name="Outcome", index=False)

    complication_table.to_excel(writer, sheet_name="Complications", index=False)

    success_log.to_excel(writer, sheet_name="Logistic_Success", index=False)

    complication_log.to_excel(writer, sheet_name="Logistic_Complications", index=False)
    
    success_logistic.to_excel(writer, sheet_name="Logistic_Success_sup", index=False)

    complication_logistic.to_excel(writer, sheet_name="Logistic_Complications_sup", index=False)
